# Cortical Wave Simulation with PyVista

This notebook demonstrates how to create an interactive 3D visualization of wave propagation across brain cortical surfaces using PyVista and FreeSurfer data. The simulation includes:

- Loading and processing FreeSurfer surface geometries (white, pial, inflated, sphere)
- Brain region parcellation and seed vertex calculation
- Synthetic wave propagation modeling with geodesic distances
- Interactive 3D visualization with time animation and surface morphing controls

## Requirements
- PyVista
- PyGeodesic 
- NiBabel
- NumPy
- Matplotlib
- FreeSurfer surface data

## 1. Import Required Libraries

Import all necessary libraries for brain surface processing, geodesic calculations, and 3D visualization.

In [1]:
# !pip install ipywidgets

In [2]:
import pyvista as pv 
import pygeodesic.geodesic as geodesic
import os

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

## 2. Set Data Paths and Load FreeSurfer Surfaces

Define the path to FreeSurfer surface files and load different surface representations:
- **White surface**: Inner cortical boundary (white matter/gray matter interface)
- **Pial surface**: Outer cortical boundary (gray matter/CSF interface)  
- **Inflated surface**: Smoothed surface for better visualization
- **Sphere surface**: Spherical representation for registration

In [5]:
surfer_path

'c:\\Users\\scedg10\\OneDrive - Cardiff University\\python\\MIC-HACK2025_CSD\\exampleSurfs'

In [7]:
import sys

# Path to your FreeSurfer surface file (e.g., lh.white)
# surfer_path = r"C:\Users\scedg10\OneDrive - Cardiff University\python\MIC-HACK2025_CSD\exampleData\HCP_rawSurfaces\100206"
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
surfer_path = os.path.join(script_dir, "exampleSurfs")
# Alternative path for different user:
# surfer_path = r"C:\Users\dgall\OneDrive - Cardiff University\python\MIC-HACK2025_CSD\exampleData\HCP_rawSurfaces\100206"

hemi = "lh"  # Left hemisphere

print(f"Loading FreeSurfer surfaces from: {surfer_path}")
print(f"Hemisphere: {hemi}")

# Load surface geometries
coords_white, faces_white = nib.freesurfer.read_geometry(os.path.join(surfer_path,"surf", f"{hemi}.white"))  
coords_pial, faces_pial = nib.freesurfer.read_geometry(os.path.join(surfer_path,"surf", f"{hemi}.pial"))  
coords_inflated, faces_inflated = nib.freesurfer.read_geometry(os.path.join(surfer_path,"surf", f"{hemi}.inflated"))
coords_inflated = coords_inflated * .55  # Scale down the inflated surface for better visualization
coords_sphere, faces_sphere = nib.freesurfer.read_geometry(os.path.join(surfer_path,"surf", f"{hemi}.sphere"))

print(f"Loaded surfaces:")
print(f"  White surface: {coords_white.shape[0]} vertices, {faces_white.shape[0]} faces")
print(f"  Pial surface: {coords_pial.shape[0]} vertices, {faces_pial.shape[0]} faces")
print(f"  Inflated surface: {coords_inflated.shape[0]} vertices, {faces_inflated.shape[0]} faces")
print(f"  Sphere surface: {coords_sphere.shape[0]} vertices, {faces_sphere.shape[0]} faces")

Loading FreeSurfer surfaces from: c:\Users\scedg10\OneDrive - Cardiff University\python\MIC-HACK2025_CSD\exampleSurfs
Hemisphere: lh
Loaded surfaces:
  White surface: 160859 vertices, 321714 faces
  Pial surface: 160859 vertices, 321714 faces
  Inflated surface: 160859 vertices, 321714 faces
  Sphere surface: 160859 vertices, 321714 faces


## 3. Load Brain Region Labels and Calculate Seed Vertices

Load the FreeSurfer parcellation labels (aparc.annot) which divide the cortex into anatomical regions. For each region, we calculate:
- Mean coordinate position of all vertices in that region
- Seed vertex (the vertex closest to the mean coordinate) for wave initialization

In [9]:
# Load brain region labels from FreeSurfer annotation file
label_file_path = os.path.join(surfer_path,"label", f"{hemi}.aparc.annot")
labels, ctab, names = nib.freesurfer.read_annot(label_file_path)

# Use mid-surface coordinates (average of pial and white matter surfaces)
coords = (coords_pial + coords_white)/2
faces = faces_pial

print(f"Loaded annotation file: {label_file_path}")
print(f"Number of unique labels: {len(np.unique(labels))}")
print(f"First 10 label names: {[name.decode('utf-8') for name in names[:10]]}")

# Calculate mean coordinates and seed vertices for each brain region
unique_labels = np.unique(labels)
label_mean_coords = {}
label_seed_vertex = {}

print("\nCalculating seed vertices for each brain region:")
for label in unique_labels:
    vertex_indices = np.where(labels == label)[0]
    mean_coord = coords[vertex_indices].mean(axis=0)
    label_mean_coords[label] = mean_coord
    
    # Find the vertex index closest to the mean coordinate
    distances = np.linalg.norm(coords[vertex_indices] - mean_coord, axis=1)
    closest_vertex = vertex_indices[np.argmin(distances)]
    label_seed_vertex[label] = closest_vertex
    
    # Print info for first few labels
    if len(label_seed_vertex) <= 5:
        print(f"  Label {label}: seed vertex {closest_vertex}, mean coord {mean_coord}")

# Create mapping from label indices to label names
label_names = {label: names[i].decode('utf-8') for i, label in enumerate(unique_labels)}

print(f"\nTotal brain regions processed: {len(unique_labels)}")
print("Label mapping created successfully.")

Loaded annotation file: c:\Users\scedg10\OneDrive - Cardiff University\python\MIC-HACK2025_CSD\exampleSurfs\label\lh.aparc.annot
Number of unique labels: 35
First 10 label names: ['unknown', 'bankssts', 'caudalanteriorcingulate', 'caudalmiddlefrontal', 'corpuscallosum', 'cuneus', 'entorhinal', 'fusiform', 'inferiorparietal', 'inferiortemporal']

Calculating seed vertices for each brain region:
  Label -1: seed vertex 90330, mean coord [-11.18169205 -12.10143407  19.75548399]
  Label 1: seed vertex 55267, mean coord [-51.44449527 -48.51657603  20.17608849]
  Label 2: seed vertex 126157, mean coord [-4.40639072 15.46528308 55.26723385]
  Label 3: seed vertex 109174, mean coord [-32.72006066  -2.44998822  70.25443997]
  Label 5: seed vertex 11131, mean coord [ -8.54267829 -86.86041709  25.1022876 ]

Total brain regions processed: 35
Label mapping created successfully.


## 4. Create Synthetic Wave Simulation

Generate a synthetic wave that propagates across the brain surface from a specific region. The wave simulation includes:
- **Wave speed**: Propagation velocity across the cortex (mm/min)
- **Wave thickness**: Duration of active wave at each location
- **Exponential decay**: Gradual fade-out after wave passage
- **Geodesic distances**: Accurate surface-based distance calculations

In [10]:
# Wave simulation parameters
wave_speed = 2  # Speed of wave in cortex (mm/min)
# Alternative realistic speed: wave_speed = 3.5 # from Hadjikhani et al. 2001

# Time parameters
timeMax = 30  # Maximum time in minutes
time_mins = np.linspace(0, timeMax, 100)
time_delta = 0.75  # Thickness of wave (in minutes)
time_decay = 2  # Decay time constant (minutes)

# Choose starting region for wave
label_index = 22  # pericalcarine region
# Alternative: label_index = 14  # lingual region

print(f"Wave simulation parameters:")
print(f"  Wave speed: {wave_speed} mm/min")
print(f"  Time range: 0-{timeMax} minutes ({len(time_mins)} time points)")
print(f"  Wave thickness: {time_delta} minutes")
print(f"  Decay time: {time_decay} minutes")
print(f"  Starting region: Label {label_index} ({label_names.get(label_index, 'Unknown')})")

# Calculate geodesic distances using PyGeodesic
print("\nCalculating geodesic distances...")
geoalg = geodesic.PyGeodesicAlgorithmExact(coords, faces)
sourceIndex = np.array([label_seed_vertex[np.where(unique_labels == label_index)[0][0]]])
targetIndex = None
distance_mm, path = geoalg.geodesicDistances(sourceIndex, targetIndex)

print(f"  Source vertex: {sourceIndex[0]}")
print(f"  Distance range: {distance_mm.min():.1f} - {distance_mm.max():.1f} mm")

# Prepare arrays for broadcasting
time_mins = np.atleast_2d(time_mins)  # Convert to row vector
distance_mm = np.atleast_2d(distance_mm).T  # Convert to column vector

# Convert distance to time of wave arrival
time_on = distance_mm / wave_speed  # Time when wave arrives (in minutes)

# Create timeseries matrix (vertices x time_points)
print("Generating wave timeseries...")
timeseries = np.zeros((len(distance_mm), len(time_mins[0])))

# Generate wave pattern for each vertex and time point
for i, t in enumerate(time_mins[0]):
    # Before wave arrives: value = 0
    mask_before = (t < time_on.flatten())
    timeseries[mask_before, i] = 0
    
    # During active wave: value = 1
    mask_active = ((t >= time_on.flatten()) & (t < (time_on.flatten() + time_delta)))
    timeseries[mask_active, i] = 1
    
    # After wave passes: exponential decay
    mask_decay = (t >= (time_on.flatten() + time_delta))
    if np.any(mask_decay):
        decay_values = np.exp(-(t - (time_on.flatten()[mask_decay] + time_delta)) / time_decay)
        timeseries[mask_decay, i] = decay_values

print(f"Timeseries shape: {timeseries.shape} (vertices x time_points)")
print(f"Value range: {timeseries.min():.3f} - {timeseries.max():.3f}")
print("Wave simulation complete!")

Wave simulation parameters:
  Wave speed: 2 mm/min
  Time range: 0-30 minutes (100 time points)
  Wave thickness: 0.75 minutes
  Decay time: 2 minutes
  Starting region: Label 22 (pericalcarine)

Calculating geodesic distances...
  Source vertex: 14492
  Distance range: 0.0 - 237.8 mm
Generating wave timeseries...
Timeseries shape: (160859, 100) (vertices x time_points)
Value range: 0.000 - 1.000
Wave simulation complete!


## 5. Build PyVista Mesh and Visualization Setup

Convert the FreeSurfer surface data to PyVista PolyData format and set up the initial 3D visualization. PyVista requires faces to be formatted with leading vertex counts (3 for triangles).

In [11]:
# Convert faces to PyVista format (add leading 3s for triangle faces)
faces_pv = np.hstack([np.full((faces_white.shape[0], 1), 3), faces_white]).astype(np.int64)

# Create PyVista mesh using mid-surface coordinates
mesh = pv.PolyData((coords_pial + coords_white)/2, faces_pv)

# Initialize mesh with first time point
mesh.point_data['timeseries'] = timeseries[:, 0]

print(f"PyVista mesh created:")
print(f"  Points: {mesh.n_points}")
print(f"  Faces: {mesh.n_faces}")
print(f"  Point data arrays: {list(mesh.point_data.keys())}")

# Create plotter with large window
# plotter = pv.Plotter(window_size=[1920, 1000])
plotter = pv.Plotter(window_size=[1200, 600])

# Set initial camera position for good brain view
plotter.camera_position = [(220, 0, 0), (-32, 0, 0), (0, 0, 1)]

print("Visualization setup complete!")
print("Mesh and plotter ready for interactive display.")

PyVista mesh created:
  Points: 160859
  Faces: 321714
  Point data arrays: ['timeseries']


c:\Users\scedg10\Anaconda3\envs\pyvista\lib\site-packages\pyvista\core\pointset.py:1386: PyVistaDeprecationWarning: The current behavior of `pv.PolyData.n_faces` has been deprecated.
                Use `pv.PolyData.n_cells` or `pv.PolyData.n_faces_strict` instead.
                See the documentation in '`pv.PolyData.n_faces` for more information.
  warnings.warn(


Visualization setup complete!
Mesh and plotter ready for interactive display.


## 6. Interactive Visualization with Sliders

Create the final interactive visualization with two main controls:
1. **Time slider**: Animate the wave propagation over time
2. **Surface morphing slider**: Morph between different surface representations:
   - 0-1: White matter → Pial surface
   - 1-2: Pial → Inflated surface
   - 2-3: Inflated → Sphere surface

The visualization uses a viridis colormap to show wave intensity across the brain surface.

In [12]:
# Define callback functions for interactive controls
def update_mesh(value):
    """Update the wave timeseries display based on time slider value"""
    global mesh
    mesh.point_data['timeseries'] = timeseries[:, int(value)]
    update_view()

def update_mesh_coords(value):
    """Update mesh coordinates to morph between different surface types"""
    global mesh
    
    if value < 0 or value > 3:
        raise ValueError("Value must be between 0 and 3")
    
    if value < 1:
        # Morph from white to pial surface
        new_coords = coords_white + (coords_pial - coords_white) * value
    elif value < 2:
        # Morph from pial to inflated surface
        new_coords = coords_pial + (coords_inflated - coords_pial) * (value - 1)
    else:
        # Morph from inflated to sphere surface
        new_coords = coords_inflated + (coords_sphere - coords_inflated) * (value - 2)
    
    mesh.points = new_coords
    update_view()
    
def update_view():
    """Refresh the mesh display"""
    plotter.add_mesh(mesh, name='brainmesh', scalars='timeseries', 
                    cmap='viridis', show_scalar_bar=False, 
                    smooth_shading=False, show_edges=False)

# Initialize the view
update_view()

# Add time slider (horizontal, top of screen)
plotter.add_slider_widget(
    lambda value: update_mesh(value),
    [0, timeseries.shape[1] - 1],
    title='Time',
    value=0,
    pointa=(0.1, 0.9, 0),
    pointb=(0.9, 0.9, 0),
    interaction_event='always',
    title_height=.03,
    style='modern'
)

# Add surface morphing slider (vertical, left side)
plotter.add_slider_widget(
    lambda value: update_mesh_coords(value),
    [0, 2],
    title='',
    value=0,
    pointa=(0.1, 0.05, 0),
    pointb=(0.1, 0.9, 0),
    interaction_event='always',
    title_height=.03,
    style='modern'
)

# Optional enhancements (uncomment to enable)
# plotter.enable_depth_of_field()
# plotter.enable_anti_aliasing('ssaa')

print("Starting interactive visualization...")
print("Controls:")
print("  - Top slider: Time animation")
print("  - Left slider: Surface morphing (white→pial→inflated→sphere)")
print("  - Mouse: Rotate, zoom, pan")

# Launch the interactive visualization
plotter.show()

Starting interactive visualization...
Controls:
  - Top slider: Time animation
  - Left slider: Surface morphing (white→pial→inflated→sphere)
  - Mouse: Rotate, zoom, pan


Widget(value='<iframe src="http://localhost:57655/index.html?ui=P_0x1c51fe95390_0&reconnect=auto" class="pyvis…

## Summary

This notebook demonstrates interactive 3D visualization of cortical wave simulations using PyVista. The visualization includes:

- **Real brain geometry**: Uses FreeSurfer surfaces (white matter, pial, inflated, and spherical)
- **Dynamic wave simulation**: Synthetic waves that propagate across the cortical surface
- **Interactive controls**: 
  - Time slider for wave animation
  - Surface morphing slider to transition between different geometric representations
- **High-quality rendering**: Smooth visualization with customizable color mapping

The simulation can be extended by:
- Loading real fMRI or MEG/EEG data instead of synthetic waves
- Adding multiple seed points for complex wave patterns
- Implementing different wave propagation models
- Customizing color schemes and rendering options

For best performance, ensure you have sufficient GPU memory and consider reducing mesh resolution for very large datasets.